# 1. Introduction

This notebook was made in the scope of the course "INFS5720".
This analysis delves into the crucial area of airline passenger satisfaction, utilizing a detailed survey dataset to understand passenger experiences and sentiment. Gaining insights into satisfaction drivers and developing predictive capabilities are essential for airlines seeking to enhance service quality and business performance.

**Task:**

The project leverages the "Airline Passenger Satisfaction" dataset, which contains survey results encompassing demographics, flight details, and numerous specific service ratings. The core tasks are to identify which factors most significantly influence whether a passenger is 'satisfied' or 'neutral or dissatisfied', and to assess the feasibility of accurately predicting this satisfaction state.

**Problem:**

The central analytical challenge is to unravel the complex interplay between diverse factors – ranging from inflight services like Wifi and seat comfort, to operational aspects like delays, and passenger characteristics like travel type or class – and their collective impact on overall satisfaction. The goal is to move beyond simple correlations to identify actionable drivers and build reliable predictive understanding.

**Objective:**

The primary objectives for this project are:

To perform comprehensive Exploratory Data Analysis (EDA) to uncover initial data patterns, distributions, relationships, and potential data quality issues.

To identify and evaluate the key factors driving passenger satisfaction through insights gained from both EDA and the application of various analytical models.

To develop, train, and rigorously evaluate multiple machine learning models (specifically Logistic Regression, Decision Tree, Random Forest, ANN, SVM, and Naive Bayes) to predict passenger satisfaction levels ('satisfied' vs. 'neutral or dissatisfied').

To compare the performance, strengths, and weaknesses of these different modeling approaches in the context of this specific problem.

**Contributers:**

Yunwei Xu z5461785

Fan Wu z5628214

Xi Lu z5319326

Jiashen Zhou z5524303

Xinyi Li z5505145

**Data Source:**

The analysis is based on the "Airline Passenger Satisfaction" dataset available on Kaggle, curated by TJ Klein: https://www.kaggle.com/datasets/teejmahal20/airline-passenger-satisfaction/data

# 2. Business Understanding

Passenger satisfaction is foundational to airline success, directly influencing crucial business outcomes. Satisfied flyers are far more likely to demonstrate loyalty through repeat bookings and program participation, act as brand advocates via positive reviews and referrals, and contribute to stable revenue streams. Conversely, dissatisfaction poses significant risks, potentially leading to customer churn, damaging negative publicity that deters new bookings, increased service recovery costs, and ultimately, an erosion of market share and profitability.

This study leverages a comprehensive airline passenger satisfaction survey dataset, which captures detailed passenger demographics, customer attributes, travel specifics, and multi-faceted service ratings. Our core objective is twofold: first, to deeply and quantitatively analyze the key factors influencing passenger satisfaction; second, to develop effective predictive models based on these factors.

To achieve these objectives, the entire analytical process – starting with Exploratory Data Analysis (EDA) and employing a suite of machine learning models (Logistic Regression, Decision Tree, eXtreme Gradient Boosting, Random Forest, ANN, SVM, Naive Bayes) – will focus on answering approximately three critical business questions:

What are the most critical factors – encompassing specific service attributes (like Wifi, seat comfort), passenger characteristics (like customer type, travel purpose, class), and operational elements (like delays) – that collectively determine overall passenger satisfaction?
Do different key passenger segments (e.g., business vs. personal travelers, loyal vs. disloyal customers, different cabin classes) exhibit distinct satisfaction drivers or patterns, suggesting a need for tailored experiences?
How accurately can we predict a passenger's satisfaction level based on these factors, and can we develop a reliable capability for proactive identification of passengers likely to be dissatisfied?
The analytical approach will systematically explore the data, using insights from EDA and cross-validating findings across diverse models – from interpretable linear and rule-based methods to robust ensembles and complex non-linear techniques – ensuring a comprehensive understanding.

Ultimately, by answering these core questions, this analysis aims to provide the airline with a clear, actionable roadmap for prioritizing service improvements, understand the nuanced needs of key customer groups, and develop proactive customer management tools. This will empower the airline to effectively enhance the overall passenger experience, strengthen customer relationships, and drive sustainable business growth.

# 3. Data Understanding (EDA)

The primary objective of this section is to obtain an initial grasp of the dataset, covering its structure, composition, and content. This preliminary understanding lays the groundwork for all the following analysis phases. According to the dataset's documentation, it is divided into two parts: 80% is designated for training, while 20% is reserved for testing.

## 3.1 Importing Required Libraries
These are whole required libraries.

In [ ]:
pip install category_encoders

In [ ]:
pip install scikeras tensorflow

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.cluster import KMeans
from xgboost import XGBClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    mean_squared_error,
    r2_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    auc,
    precision_recall_curve,
    make_scorer
)
from sklearn.svm import SVC
from IPython.display import Image
from sklearn.tree import plot_tree
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint
import matplotlib.gridspec as gridspec
from matplotlib import cm
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import plot_model
import category_encoders as ce
import warnings
warnings.filterwarnings("ignore")

## 3.2 Loading the Dataset

We directly use the training dataset and test dataset divided by former colleagues. EDA currently only explores the training dataset. The test dataset remains unused during EDA phase to prevent data leakage and affect the model's performance evaluation.

In [ ]:
# Load datasets (download from Kaggle: https://www.kaggle.com/datasets/teejmahal20/airline-passenger-satisfaction)
import pandas as pd
df = pd.read_csv('data/train.csv')
df_test = pd.read_csv('data/test.csv')
print("Train shape:", df.shape)
print("Test shape:", df_test.shape)

In [ ]:
df = pd.read_csv('train.csv')

print("Data shape:", df.shape)


print("\nFirst 5 rows of data:")
print(df.head())


print("\nData information:")
df.info()


print("\nColumn names:", df.columns)


print(f"\nNumber of duplicate rows: {df.duplicated().sum()}")


## 3.3 Handle the dataset

Calculate the number and proportion of missing values in each column.

In [ ]:
# Calculate the number of missing values in each column.
missing_values = df.isnull().sum()
print("\nNumber of missing values in each column:")
print(missing_values[missing_values > 0]) # Only show columns containing missing values.

# Calculate the proportion of missing values in each column.
missing_percentage = (df.isnull().sum() / len(df)) * 100
print("\nProportion of missing values in each column (%):")
print(missing_percentage[missing_percentage > 0].round(2)) # Only show columns containing missing values.

This step removes the 'Unnamed: 0' and 'id' columns, which are irrelevant for analysis, and drops rows with missing values in 'Arrival Delay in Minutes' since their proportion is small.

In [ ]:
df = df.drop(columns=['Unnamed: 0', 'id'])
df = df.dropna(subset=['Arrival Delay in Minutes'])

In [ ]:
print(df.head())

## 3.4 EDA Data Visualization

### 3.4.1 Target Distribution

In [ ]:
# 1. Calculate the distribution of satisfaction labels
satisfaction_counts = df['satisfaction'].value_counts()

# 2. Prepare plotting data
labels_for_legend = satisfaction_counts.index
sizes = satisfaction_counts.values
total = sizes.sum()

# 3. Define the autopct formatting function to display both percentage and count.
#    p is the percentage of the current sector.
#    count = p / 100 * total
def format_autopct(p):
    value = int(round(p * total / 100.0))
    return f'{p:.1f}%\n({value})' # format

# 4. Draw a pie chart.
plt.figure(figsize=(9, 7))
colors = sns.color_palette('viridis', len(labels_for_legend))
wedges, texts, autotexts = plt.pie(sizes,
                                   autopct=format_autopct,
                                   startangle=90,
                                   colors=colors,
                                   pctdistance=0.75,
                                   wedgeprops={'edgecolor': 'white'})

plt.setp(autotexts, size=10, weight="bold", color="white")

# 5. Add a legend to identify categories.
plt.legend(wedges, labels_for_legend,
           title="Satisfaction Category",
           loc="center left",
           bbox_to_anchor=(1, 0, 0.5, 1))

plt.title('Distribution of Satisfaction (% and Count)')
plt.axis('equal')
plt.show()

The preliminary analysis reveals the overall state of passenger satisfaction. The target variable, Satisfaction, includes two categories: "satisfied" and "neutral or dissatisfied". Out of a total of 103904 passengers, approximately 56.7% were classified as "neutral or dissatisfied", while 43.3% of passengers were labelled as "satisfied". This distribution is visualized using a pie chart that clearly displays that the 'neutral or dissatisfied' category represents more than half of the passengers.

This represents potential risks of customer churn, negative word-of-mouth, and an urgent need for service improvements.

Because of the imbalance, model evaluation should prioritise metrics such as F1-Score and ROC AUC, rather than relying solely on accuracy, to better capture the balance between precision and recall.

### 3.4.2 Descriptive Analysis of Numerical and Categorical Features

In [ ]:
# Select columns of numerical type (including pseudo-numerical columns for ratings).
numerical_cols = df.select_dtypes(include=np.number).columns.tolist()

print("\nList of numerical features:", numerical_cols)

# Obtain descriptive statistical information of numerical features.
print("\nDescriptive statistics of numerical features:")
print(df[numerical_cols].describe().round(2))

# Draw the distribution diagram (histogram/kernel density estimation + box plot) for each numerical feature.
print("\nDraw the distribution map of numerical features:")
for col in numerical_cols:
    plt.figure(figsize=(12, 4))

    # Histogram and Kernel Density Estimate
    plt.subplot(1, 2, 1)
    sns.histplot(data=df, x=col, kde=True, bins=30) # bins can be adjusted
    plt.title(f'Distribution of {col}')

    # Boxplot (to observe median, quartiles, and outliers)
    plt.subplot(1, 2, 2)
    sns.boxplot(y=df[col])
    plt.title(f'Boxplot of {col}')

    plt.tight_layout()
    plt.show()

In this step, we conducted a statistical summary and visualisation of 18 numerical features in the training dataset. These features encompass basic passenger information, flight characteristics, 14 service rating items, and operational performance (such as delay durations).

The output shows that the average passenger age is approximately 39.4 years, with a standard deviation of 15.1, indicating a wide age range (from 7 to 85 years old). The average flight distance is 1,189 miles, but with a standard deviation close to 1,000 miles and a maximum value nearing 5,000 miles, the distribution is heavily right-skewed. This suggests that most trips are short to medium-haul, with only a few long-haul flights. Such a wide age range implies that service design must account for varying preferences across age groups. The significant variation in travel distances indicates that passengers may have different needs and sensitivities toward in-flight amenities such as comfort and entertainment, particularly for long-haul travellers, whose experience carries greater weight.

Among the 14 service rating features, the mean scores range from 2.73 to 3.64. Inflight Wi-Fi service (mean: 2.73) and Ease of online booking (mean: 2.76) have the lowest average ratings, highlighting them as potential problem areas. In contrast, Baggage handling (mean: 3.63) and in-flight service (mean: 3.64) received the highest average ratings. Most features have a median rating of 3 or 4. This suggests that Wi-Fi access and the online booking process are significant weaknesses and should be prioritised for improvement. Meanwhile, baggage handling and in-flight service staff performance are relative strengths. Although many ratings fall within the acceptable range, several aspects still warrant further enhancement.

The average departure and arrival delay is around 15 minutes, but the standard deviation is large (about 38 minutes), with extreme values exceeding 1,500 minutes. Both medians are 0 minutes, and the 75th percentile is around 12–13 minutes, indicating a highly right-skewed distribution. This suggests that while the vast majority of flights are on time or experience only minor delays—a positive for most passengers—a small number of extreme delays heavily inflate the average. These outliers likely result in very negative passenger experiences and represent a serious reputational risk. It’s crucial to investigate the frequency and causes of these extreme delays (e.g., severe weather) or check for potential data entry errors.

In [ ]:
# Select columns of object/categorical type (excluding the target variable 'Satisfaction').
categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
if 'Satisfaction' in categorical_cols:
    categorical_cols.remove('Satisfaction')

print("\nCategory feature list:", categorical_cols)

# Draw a count plot for each category feature.
print("\nDraw a category feature count plot.")
for col in categorical_cols:
    plt.figure(figsize=(8, 5))
    # Order bars by frequency
    value_order = df[col].value_counts().index
    sns.countplot(data=df, x=col, order=value_order, palette='viridis')
    plt.title(f'Distribution of {col}')
    # Rotate x-axis labels if they are long or numerous
    if len(value_order) > 5:
         plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

    # Print proportions for each category
    print(f"\nValue Counts for {col} (%):\n", (df[col].value_counts(normalize=True) * 100).round(2))

The distribution of the Gender feature in the sample is fairly balanced, with female passengers accounting for approximately 50.8% and male passengers around 49.3%, reflecting an almost 1:1 ratio. This indicates that there is no gender bias in the data collection process, and thus no gender-specific balancing is required during analysis.

The Customer Type feature exhibits a significant imbalance, with “Loyal customers” making up the majority (approximately 81.7%), while “Disloyal customers” account for only around 18.3%. This disparity suggests that the majority of passengers are regular users of the airline.

Regarding Type of Travel, “Business travel” dominates the dataset with around 69%, significantly higher than “Personal travel” at 31%. Business travellers form the primary customer base, and their needs should be the central focus of service optimisation. Their more predictable and uniform expectations allow the company to streamline operations and optimise service offerings more efficiently. While personal travellers form a smaller segment, they still represent an important group that may require differentiated services to address more diverse or flexible preferences.

In terms of Class, “Business class” has the highest proportion, at around 47.9%, followed by “Eco” (economy class) at approximately 45%, while “Eco Plus” accounts for just 7.2%. Business and economy classes make up the bulk of the passenger base.

In summary, the passenger profile depicted in the sample is: gender-balanced, predominantly loyal customers, mainly business travellers, and roughly equal representation between business and economy class, with a slight tilt toward business class.



The next critical step is to link these categorical features to the satisfaction target variable, i.e., perform cross-tabulation and comparative analysis. The goal is to identify whether there are statistically significant differences in satisfaction across different customer types, travel purposes, and class levels, ultimately uncovering the key categorical dimensions that influence customer satisfaction.


### 3.4.3 Categorical Feature Analysis for Satisfaction Differences

In [ ]:
print("\nAnalyze the relationship between features and satisfaction.")

# a) Numerical Features vs. Satisfaction
print("\nNumerical Features vs Satisfaction:")
for col in numerical_cols:
    plt.figure(figsize=(10, 5))

    # Compare distributions using boxplot
    plt.subplot(1, 2, 1)
    sns.boxplot(data=df, x='satisfaction', y=col, palette='viridis')
    plt.title(f'{col} by satisfaction (Boxplot)')

    plt.subplot(1, 2, 2)
    sns.violinplot(data=df, x='satisfaction', y=col, palette='viridis')
    plt.title(f'{col} by satisfaction (Violinplot)')

    plt.tight_layout()
    plt.show()

    print(f"\nMean of {col} by satisfaction:\n", df.groupby('satisfaction')[col].mean().round(2))
    print(f"Median of {col} by satisfaction:\n", df.groupby('satisfaction')[col].median().round(2))


# b) Categorical Features vs. Satisfaction
print("\nCategory Feature vs Satisfaction:")
for col in categorical_cols: # Already excluded 'Satisfaction' earlier
    plt.figure(figsize=(10, 6))
    # Use grouped countplot to show Satisfaction distribution within each category
    value_order = df[col].value_counts().index # Keep consistent ordering
    sns.countplot(data=df, x=col, hue='satisfaction', order=value_order, palette='viridis')
    plt.title(f'{col} vs satisfaction')
    if len(df[col].unique()) > 5:
        plt.xticks(rotation=45, ha='right')
    plt.legend(title='satisfaction')
    plt.tight_layout()
    plt.show()

    crosstab_norm = pd.crosstab(df[col], df['satisfaction'], normalize='index') # Normalize by row
    print(f"\nsatisfaction Rate within {col}:\n", (crosstab_norm * 100).round(2))

Most service rating features—including Inflight wifi service, Food and drink, Online boarding, Seat comfort, Inflight entertainment, On-board service, Leg room service, Checkin service, Inflight service, and Cleanliness—show significantly higher mean and median scores among satisfied passengers compared to the neutral or dissatisfied group. The differences are particularly notable for Online boarding, Seat comfort, and in-flight entertainment. This highlights that the core in-flight experience is the key battleground for customer satisfaction. Aspects such as the ease of the online boarding process, seat comfort, variety of inflight entertainment, and cabin cleanliness are central to passengers' perceptions of service quality.

Previously identified weak points—wifi service and online booking—also show strong correlations with satisfaction here, reinforcing their role as urgent priorities for improvement with high potential return.

The data also reveals that satisfied passengers tend to be older (40-60) and fly longer distances. While both groups have a median delay of 0 minutes, the average delay is slightly lower among satisfied customers, suggesting that extreme delays are more common among the dissatisfied group. Features like Departure/Arrival time convenience and Gate location show little to no correlation with satisfaction. The higher satisfaction levels among older and long-haul passengers may be linked to their travel purposes (e.g., business), preferred cabin class, or differing service expectations—areas worth further exploration. While most passengers experience minimal delays, this analysis underscores the importance of managing flight delays, especially severe ones, to preserve customer satisfaction.

Three categorical features—Customer Type, Type of Travel, and Class—exhibit very strong associations with satisfaction. Loyal customers, business travellers, and business class passengers show much higher satisfaction levels than disloyal customers, personal travellers, or economy/super economy passengers. In contrast, Gender shows only a weak relationship with satisfaction.

These findings suggest that loyalty status, travel purpose, and cabin class are the three most decisive predictors of satisfaction. While maintaining high satisfaction among core segments (loyal customers, business travellers, business class) is crucial, the extremely low satisfaction levels among personal travellers and economy passengers (with personal travel satisfaction as low as 10%) raise red flags. This may indicate serious issues in services aimed at the broader customer. Increasing conversion rates among disloyal customers also emerges as a key strategic challenge.

This analysis has identified numerous features strongly associated with satisfaction, including most service ratings, customer type, travel purpose, cabin class, flight distance, and age. These will serve as strong candidates for predictive modelling. Meanwhile, features such as gender, gate location, and time convenience appear to have limited predictive power. A clear picture of the key drivers of satisfaction is now in place. The next stage will focus on quantifying the relative importance of these factors and predicting which passengers are likely to be dissatisfied or satisfied.


### 3.4.4 Correlation Analysis

In [ ]:
try:
    # 1. Encode the target variable
    df['Satisfaction_encoded'] = df['satisfaction'].map({'satisfied': 1, 'neutral or dissatisfied': 0})

    # 2. One-hot encode categorical features and obtain new column names
    categorical_to_encode = ['Class', 'Type of Travel']
    df_encoded = pd.get_dummies(df, columns=categorical_to_encode, prefix=['Class', 'TravelType'], drop_first=False)

    encoded_categorical_cols = [col for col in df_encoded.columns if col.startswith('Class_') or col.startswith('TravelType_')]

    # 3. Prepare the final list of columns for calculating correlation.
    if 'Satisfaction_encoded' not in df_encoded.columns:
        raise ValueError("'Satisfaction_encoded' column not created.")
    if not encoded_categorical_cols:
         raise ValueError("Dummy columns for categoricals not created.")

    cols_to_correlate = numerical_cols + encoded_categorical_cols + ['Satisfaction_encoded']

    # 4. Calculate the correlation matrix
    print(f"Calculating correlation for {len(cols_to_correlate)} columns...")
    correlation_matrix_full = df_encoded[cols_to_correlate].corr()

    # 5. Draw heat map
    print("Plotting heatmap...")
    num_features = len(cols_to_correlate)
    fig_width = max(18, num_features * 0.7)
    fig_height = max(15, num_features * 0.6)
    annot_size = max(5, 10 - num_features // 4)

    plt.figure(figsize=(fig_width, fig_height))
    sns.heatmap(correlation_matrix_full, annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5, annot_kws={"size": annot_size})
    plt.title('Correlation Matrix (Numerical + OHE Categorical + Target)', fontsize=14)
    plt.xticks(rotation=50, ha='right', fontsize=8)
    plt.yticks(rotation=0, fontsize=8)
    plt.tight_layout(pad=1.0)
    plt.show()

except KeyError as e:
    print(f"Error: Column not found - {e}. Please ensure df, numerical_cols, 'satisfaction', 'Class', 'Type of Travel' exist.")
except ValueError as e:
    print(f"Error: {e}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

In [ ]:
target_correlations = correlation_matrix_full['Satisfaction_encoded'].drop('Satisfaction_encoded').sort_values(ascending=False)

if 'target_correlations' in locals():
    print("\nVisualizing Feature Correlations with Target as a Heatmap:")

    # 1.Convert Series to DataFrame (single column)
    target_corr_df = target_correlations.to_frame(name='Correlation with Satisfaction')

    # 2. Draw a heatmap of this single-column DataFrame.
    plt.figure(figsize=(6, 12))
    sns.heatmap(target_corr_df,
                annot=True,
                cmap='coolwarm',
                fmt=".2f",
                linewidths=.5,
                linecolor='lightgray',
                annot_kws={"size": 10},
                vmin=-1, vmax=1,
                cbar=True)

    plt.title('Feature Correlation with Encoded Satisfaction', fontsize=14)
    plt.xticks([])
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

else:
    print("Error: 'target_correlations' Series not found. Please run the correlation calculation first.")


This step calculates and visualises the Pearson correlation coefficients between numerical features, one-hot encoded categorical features (Class, Type of Travel), and the encoded target variable (Satisfaction_encoded). The goal is to identify multicollinearity among predictors and their direct linear associations with satisfaction.

Most in-flight service ratings show moderate positive correlations (+0.3 to +0.7), particularly for core experience items (comfort, entertainment, cleanliness) and between Wi-Fi and online booking (+0.72). The widespread positive correlations among service ratings confirm the holistic nature of passenger experience perception—a single aspect's quality may influence the overall evaluation. The correlation between Wi-Fi and online booking likely reflects passengers' perception of consistency in digital experiences.

Regarding correlations with Satisfaction_encoded, Class_Business (+0.50), Online boarding (+0.50), and TravelType_Business travel (+0.45) are the strongest positive correlates, while Class_Eco (-0.45) and TravelType_Personal Travel (-0.45) are the strongest negative correlates. Core service ratings such as Inflight entertainment (+0.40), Seat comfort (+0.35), and Cleanliness (+0.31) also show significant positive correlations. This shows that cabin class, online boarding experience, and travel purpose are critical distinguishing factors for satisfaction. Core in-flight service quality (entertainment, comfort, cleanliness) is strongly positively correlated with satisfaction, reinforcing its importance.

This whole process confirms the dominant linear association of Class, Online Boarding, and Travel Type with satisfaction, while reinforcing the importance of service quality. It also flags potential multicollinearity requiring attention in certain models. The correlations among service ratings and their associations with the target indicate that feature selection, regularisation, or dimensionality reduction may benefit certain models.


## 3.5 Import and Handle the Test Dataset

Import and preprocess the test dataset for subsequent model use.

In [ ]:
# Load datasets (download from Kaggle: https://www.kaggle.com/datasets/teejmahal20/airline-passenger-satisfaction)
import pandas as pd
df = pd.read_csv('data/train.csv')
df_test = pd.read_csv('data/test.csv')
print("Train shape:", df.shape)
print("Test shape:", df_test.shape)

In [ ]:
df_test= pd.read_csv('test.csv')
print("Data shape:", df_test.shape)
print(df_test.head())
print(df_test.info())

In [ ]:
df_test= df_test.drop(columns=['Unnamed: 0', 'id'])
df_test= df_test.dropna(subset=['Arrival Delay in Minutes'])

In [ ]:
print(df_test.head())

# 4. Model Applying

## 4.1 Naive Bayes





In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score, roc_curve
import matplotlib.pyplot as plt
import seaborn as sns

### 4.1.1  Evaluate Model Performance

In [ ]:
df_NB = df.copy()
df_test_NB = df_test.copy()

# Label encoding target variable in df_NB train set
df_NB['satisfaction'] = df_NB['satisfaction'].map({'satisfied': 1, 'neutral or dissatisfied': 0})

# Feature Encoding
cat_cols = df_NB.select_dtypes(include=['object']).columns
df_encoded = df_NB.copy()

le = LabelEncoder()
for col in cat_cols:
    df_encoded[col] = le.fit_transform(df_NB[col])



# Label encoding target variable in df_test_NB test set
df_test_NB['satisfaction'] = df_test_NB['satisfaction'].map({'satisfied': 1, 'neutral or dissatisfied': 0})

# Feature Encoding
cat_cols = df_test_NB.select_dtypes(include=['object']).columns
df_test_encoded = df_test_NB.copy()

le = LabelEncoder()
for col in cat_cols:
    df_test_encoded[col] = le.fit_transform(df_test_NB[col])


X_train = df_encoded.drop('satisfaction', axis = 1)
y_train = df_encoded['satisfaction']

X_test = df_test_encoded.drop('satisfaction', axis = 1)
y_test = df_test_encoded['satisfaction']

# Standardized numerical characteristics (optional)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Model training
nb_model = GaussianNB()
nb_model.fit(X_train_scaled, y_train)

# Prediction and Evaluation
y_pred = nb_model.predict(X_test_scaled)
y_proba = nb_model.predict_proba(X_test_scaled)[:,1]
y_pred_classes = (y_pred > 0.5).astype(int)

# Accuracy & report
print("📊 Accuracy:", accuracy_score(y_test, y_pred))
print("\n📋 Classification Report:\n", classification_report(y_test, y_pred))

# AUC-ROC curve
auc = roc_auc_score(y_test, y_proba)
fpr, tpr, thresholds = roc_curve(y_test, y_proba)

plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, label=f"AUC = {auc:.4f}")
plt.plot([0,1], [0,1], 'k--')
plt.title("Naive Bayes - ROC Curve")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.grid(True)
plt.show()



### 4.1.2 Naive Bayes Confusion Matrix

In [ ]:
# Confusion Matrix
plt.figure(figsize=(6,4))
sns.heatmap(confusion_matrix(y_test, y_pred_classes), annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

Insights of Naive Bayes

We first tried Gaussion Naive Bayes as a speed benchmark which has extremely low computational costs to explore how the correlation between features affect the result in this dataset. It assumes that all factors independently affect satisfaction and compute at a rapid pace.

The performance of Naive Bayes seems good with an accuracy of 86% and AUC = 0.92, indicating that the impact of various service factors on satisfaction is relatively independent, or its independence assumption is not significant in this scenario. But the accuracy still has space to improve which states the importance of applying other models. The performance gap between Naive Bayes and other models can quantify the importance of the correlation between features, understand how much prediction accuracy is lost by ignoring feature correlations.


## 4.2 Logistic Regression

### 4.2.1 Evaluate Model Performance

4.2.1.1 Basic model performance

In [ ]:
df_LR = df.copy()
df_LR_test=df_test.copy()


df_LR['satisfaction'] = (df_LR['satisfaction'] == 'satisfied').astype(int)
df_LR_test['satisfaction'] = (df_LR_test['satisfaction'] == 'satisfied').astype(int)


X_train = df_LR.drop(columns='satisfaction')
y_train = df_LR['satisfaction']

X_test = df_LR_test.drop(columns='satisfaction')
y_test = df_LR_test['satisfaction']


X_train_encoded = pd.get_dummies(X_train, drop_first=True)
X_test_encoded = pd.get_dummies(X_test, drop_first=True)


X_test_encoded = X_test_encoded.reindex(columns=X_train_encoded.columns, fill_value=0)


model = LogisticRegression(max_iter=1000)
model.fit(X_train_encoded, y_train)


y_pred = model.predict(X_test_encoded)
print("📊 Classification Report:\n")
print(classification_report(y_test, y_pred))


4.2.1.2 Optimal model performance

In [ ]:
# 1. Copy the original data
train_LR = df.copy()
test_LR = df_test.copy()

# 2. Encode 'satisfaction' column to binary labels
label_map = {'neutral or dissatisfied': 0, 'satisfied': 1}
train_LR['satisfaction'] = train_LR['satisfaction'].map(label_map)
test_LR['satisfaction'] = test_LR['satisfaction'].map(label_map)

# 3. Drop rows with missing values
train_LR = train_LR.dropna()
test_LR = test_LR.dropna()

# 4. One-hot encode categorical variables
x_train = pd.get_dummies(train_LR.drop("satisfaction", axis=1), drop_first=True)
x_test = pd.get_dummies(test_LR.drop("satisfaction", axis=1), drop_first=True)

# 5. Ensure train and test have the same columns
x_test = x_test.reindex(columns=x_train.columns, fill_value=0)

# 6. Extract target variable
y_train = train_LR["satisfaction"]
y_test = test_LR["satisfaction"]

# 7. Train logistic regression model
model = LogisticRegression(max_iter=1000)
model.fit(x_train, y_train)

# 8. Predict and evaluate
y_pred = model.predict(x_test)
print("📊 Classification Report:\n")
print(classification_report(y_test, y_pred, target_names=["neutral/dissatisfied", "satisfied"]))



### 4.2.2 Extension of Optimal Model (Logit)

In [ ]:
df_LR = df.copy()
df_LR = df_LR.dropna().copy()


df_LR['satisfaction'] = (df_LR['satisfaction'] == 'satisfied').astype(int)


X = df_LR.drop(columns='satisfaction')
y = df_LR['satisfaction']


X = pd.get_dummies(X, drop_first=True)
X = X.loc[:, X.nunique() > 1]


X = X.astype(float)


X = X.replace([np.inf, -np.inf], np.nan).dropna()
y = y.loc[X.index]

#VIF
vif_df = pd.DataFrame()
vif_df["feature"] = X.columns
vif_df["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
X = X[vif_df[vif_df["VIF"] < 10]["feature"]]


X_scaled = StandardScaler().fit_transform(X)
X_const = sm.add_constant(X_scaled)





model = sm.Logit(y, X_const)
result = model.fit()


coef_df = pd.DataFrame({
    'Variable': ['const'] + list(X.columns),
    'Coefficient': result.params,
    'P-value': result.pvalues
})

sig_vars = coef_df[coef_df['P-value'] < 0.05]
sig_vars_sorted = sig_vars.reindex(sig_vars['Coefficient'].abs().sort_values(ascending=False).index)


print(f"\n✅ Pseudo R-squared: {result.prsquared:.4f}")
print(f"✅ LLR p-value: {result.llr_pvalue:.4g}")
print("\n✅ Significant Variables (p < 0.05):")
print(sig_vars_sorted)


In [ ]:
# rank p values
sig_vars_sorted = sig_vars_sorted.sort_values(by='P-value', ascending=True)
sig_vars_sorted

The logistic regression model achieved a pseudo R-squared of 0.506, indicating a strong model fit for a classification task.
A total of 22 variables were statistically significant (p < 0.05), showing a diverse set of predictors contributing to satisfaction.

Top Positive Predictors:

Type of Travel (coef = +1.41): Business travelers are much more likely to report satisfaction.

Online Boarding, Inflight Wifi, Checkin Service, and On-board Service all had strong positive effects, showing that digital convenience and inflight experience are key drivers of satisfaction.

 Notable Negative Predictors:

Arrival Delay in Minutes (coef = –0.35): Unsurprisingly, delays significantly reduce satisfaction.

Departure Time Convenience and Ease of Online Booking also negatively affected satisfaction, highlighting the need for better planning tools and a smoother booking experience.

Overall, the results emphasize that both functional service (boarding, delay, checkin) and emotional/user experience (wifi, seat comfort) matter in shaping customer satisfaction.

Model Performance Comparison: Before vs After Improvements
In the initial logistic regression model (second image), the overall accuracy reached 80%, with a precision of 0.88 for dissatisfied passengers and recall of 0.87 for satisfied passengers. However, there was an imbalance: the model was biased toward predicting dissatisfaction, leading to lower recall for neutral/satisfied passengers.

After improvements (first image), including feature selection, data cleaning, and model refinement, the accuracy increased slightly to 81%, but more importantly, precision and recall became more balanced across both classes:

Class 0 (Dissatisfied):

Precision dropped slightly from 0.88 → 0.84

Recall improved from 0.75 → 0.81

Class 1 (Satisfied):

Precision improved from 0.73 → 0.76

Recall slightly dropped from 0.87 → 0.80

The final model demonstrates better generalization, avoids overfitting to one class, and achieves higher macro and weighted averages (0.80–0.81). This suggests a more reliable and fairer prediction performance, especially for business use cases involving customer satisfaction classification.

Takeaway                                        |                        Action
🔔 25% of dissatisfied customers are not caught | Implement feedback prompts post-service to catch unhappy flyers missed by the model
🟢 Satisfied customers are well identified (87% recall) | Use this to confidently segment for upselling or advocacy campaigns
⚖️ Balanced overall performance (F1 ≈ 0.80) | Great for initial deployment or integration into a larger CRM pipeline

### 4.2.3 Supplement of Logistic Regression (Radar Map Using K-Means)

In [ ]:
target_col = df_LR.columns[-1]

）
service_cols = [
    'Inflight wifi service', 'Departure/Arrival time convenient',
    'Ease of Online booking', 'Gate location', 'Food and drink',
    'Online boarding', 'Seat comfort', 'Inflight entertainment',
    'On-board service', 'Leg room service', 'Baggage handling',
    'Checkin service', 'Inflight service', 'Cleanliness'
]
features = df_LR[service_cols].dropna()


#  KMeans

scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)

kmeans = KMeans(n_clusters=3, random_state=42)
df = df.loc[features.index]
df['Cluster'] = kmeans.fit_predict(features_scaled)


cluster_means = df.groupby('Cluster')[service_cols].mean()


labels = service_cols
num_vars = len(labels)

angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
labels += [labels[0]]
angles += [angles[0]]


fig, ax = plt.subplots(figsize=(9, 7), subplot_kw=dict(polar=True))
colors = ['red', 'green', 'blue']


for i, (index, row) in enumerate(cluster_means.iterrows()):
    values = row.tolist()
    values += [values[0]]
    ax.plot(angles, values, label=f'Cluster {index}', color=colors[i], linewidth=2)
    ax.fill(angles, values, alpha=0.25, color=colors[i])


ax.set_thetagrids(np.degrees(angles[:-1]), labels[:-1], fontsize=10)
ax.set_title(" Service Rating Radar per Cluster", fontsize=14,pad=30)
ax.legend(loc='upper right', bbox_to_anchor=(1.2, 1.05))
plt.tight_layout()
plt.show()

**Cluster 1: Loyal Advocates (Green)**
Characteristics:

Highest average ratings across all service dimensions.

Particularly satisfied with:

 Online Boarding

Cleanliness

Seat Comfort

Recommendations:

These are your most loyal and satisfied customers.

Engage them in referral programs, loyalty campaigns, or customer interviews.

Maintain high service standards—any drop in quality may risk losing this valuable group.

**Cluster 0: Practical Neutrals (Red)**
Characteristics:

Satisfied with core travel experiences like seat comfort, entertainment, and boarding.

Lower ratings for:

In-flight WiFi

Food & Drink

Departure/Arrival Time Convenience

Recommendations:

Focus on enhancing digital and schedule-related services.

Consider personalized meal options or basic service upgrades to improve perceptions.

Promotional campaigns or limited-time offers may nudge them toward higher satisfaction.

**Cluster 2: At-Risk Detractors (Blue)**
Characteristics:

Lowest satisfaction scores across all categories.

Particularly low on:

In-flight WiFi

Food & Drink

Seat Comfort

Recommendations:

These customers are at high risk of churn.

Roll out targeted compensation strategies (vouchers, upgrades, service surveys).

Consider whether they are price-sensitive or on shorter/less-serviced flights—segment accordingly.

Important to listen and act fast on their feedback.

Given the model’s strong performance (F1-score = 0.96), it is highly suitable for deployment in a real-world airline customer experience system.
Below are suggested applications and integration pathways:

1. Customer Satisfaction Prediction Engine
Use Case: Automatically flag passengers likely to be dissatisfied based on their booking, flight, and service records.

Action: Trigger proactive service recovery actions (e.g., send apology emails, offer vouchers, or assign dedicated support).

2. Personalized Service Improvements
Use Case: Tailor inflight services (e.g., wifi access, seat upgrades) or boarding experiences based on predicted satisfaction probability.

Action: Provide real-time suggestions to cabin crew or at check-in to improve passenger experience.

3. Dashboard Integration for CX Teams
Use Case: Visualize satisfaction risk levels across flights, customer types, and service features.

Action: Enable data-driven prioritization of which routes, classes, or customer segments to improve first.

 4. Feedback Loop & Continuous Learning
Use Case: Feed post-flight satisfaction surveys back into the model pipeline.

Action: Improve accuracy over time by retraining on new customer behavior trends (e.g., changing wifi expectations or online boarding adoption).

## 4.3. Decision Tree

In [ ]:
train_decisiontree = df.copy()
test_decisiontree = df_test.copy()

# Delete 'id' and 'Age' columns from train and test
train = train_decisiontree.drop(columns=['id', 'Age'], errors='ignore')
test = test_decisiontree.drop(columns=['id', 'Age'], errors='ignore')

In [ ]:
# Encoding strings in each attribute
label_encoders = {}
for col in train.select_dtypes(include='object').columns:
    le = LabelEncoder()
    train[col] = le.fit_transform(train[col].astype(str))
    if col in test.columns:
        test[col] = le.transform(test[col].astype(str))
    label_encoders[col] = le

# split the predictor and other attribute
target_column = 'satisfaction'
X_train = train.drop(columns=[target_column])
y_train = train[target_column]

X_val = test.drop(columns=[target_column])
y_val = test[target_column]

### 4.3.1 Basic Model of Decision Tree

In [ ]:
#Basic Decision Tree Model

# Assuming X_train, X_test, y_train, y_test are ready (with preprocessing)
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train, y_train)

y_pred_dt = dt_model.predict(X_val)

print("Decision Tree Accuracy:", accuracy_score(y_val, y_pred_dt))
print("Classification Report (Decision Tree):\n", classification_report(y_val, y_pred_dt))


### 4.3.2 Hyperparameter Fine-tuning of Decision Tree

4.3.2.1 model evaluation and feature importance exploration

In [ ]:
# set parameter for Gridsearchcv
param_grid = {
    'max_depth': [3, 5, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'criterion': ['gini', 'entropy']
}

# Model training and try different parameter
clf = DecisionTreeClassifier(random_state=42)
grid_search = GridSearchCV(clf, param_grid, cv=5, scoring='roc_auc', n_jobs=-1)
grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_val)
y_prob = best_model.predict_proba(X_val)[:, 1]

# Model evaluation
print("Best Parameter:", grid_search.best_params_)
print("Accuracy:", accuracy_score(y_val, y_pred))
print("ROC AUC:", roc_auc_score(y_val, y_prob))
print("Confusion Matrix:\n", confusion_matrix(y_val, y_pred))
print("Classification Report:\n", classification_report(y_val, y_pred))

# Ploting Feature Importance
importances = pd.Series(best_model.feature_importances_, index=X_train.columns)
top_features = importances.sort_values(ascending=False).head(15)

plt.figure(figsize=(8, 5))
top_features.plot(kind='barh')
plt.title("Top 15 Feature Importances")
plt.xlabel("Importance")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

# Ploting Predicted Probability
plt.figure(figsize=(6,6))
plt.scatter(y_val, y_prob, c=y_val, cmap='coolwarm', alpha=0.6)
plt.plot([0, 1], [0, 1], 'r--')
plt.xlabel("Actual")
plt.ylabel("Predicted Probability")
plt.title("Actual vs Predicted Probability")
plt.grid(True)
plt.tight_layout()
plt.show()

# ROC Curve
fpr, tpr, thresholds = roc_curve(y_val, y_prob)
plt.figure(figsize=(6,6))
plt.plot(fpr, tpr, label=f'AUC = {roc_auc_score(y_val, y_prob):.2f}')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

4.3.2.2 Confusion Matrix of Decision Tree tuned model

In [ ]:
# Confusion Matrix
plt.figure(figsize=(6,4))
sns.heatmap(confusion_matrix(y_val, y_pred), annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

**Analyais & Insights**

We tried Decision Tree Model with GridSearchCV to tune the best model parameters. The AUC score is 0.99, which means that the model has excellent ability to distinguish between the satisfied and unsatisfied passengers

1 Model Evaluation Summary

The tuned Decision Tree model achieved an overall accuracy of 95% and a macro-averaged F1 score of 0.95, indicating balanced performance across both satisfied and unsatisfied customer classes. The recall for the 'Satisfied' class reached 0.92, reflecting the model's strong ability to correctly identify satisfied customers. Moreover, the precision for the 'Satisfied' class improved to 0.96, reducing the likelihood of false positives.

The ROC AUC score of 0.988 further highlights the model's excellent discrimination capability across classification thresholds. Compared to the baseline version, the tuned model demonstrates enhanced generalization, with improved sensitivity and precision, making it more reliable for practical deployment in satisfaction prediction tasks. However, as a single-tree model, it may still be more prone to overfitting compared to ensemble approaches like Random Forest.

2 Insights from feature importance

According to the feature importance diagram, Online boarding is the most influential factor in predicting customer satisfaction, accounting for over 35% of the model’s decision-making process. This highlights the importance of a seamless and efficient boarding process in shaping positive passenger experiences.

Inflight Wi-Fi service and Type of Travel follow as the second and third most important features, emphasizing the significance of digital connectivity and the context of travel (e.g., business vs Personal) in determining satisfaction.

Other features include Inflight entertainment, Class, and Customer Type, reflecting passenger sensitivity to comfort, service differentiation, and consistency. In contrast, operational metrics such as Flight distance and Arrival delay show relatively low influence, suggesting that service quality is a more critical satisfaction driver than travel time or distance.

### 4.3.3 Supplement for Decision Tree——eXtreme Gradient Boosting

In [ ]:
df_LR = df.copy()
df_LR.drop(columns=["Unnamed: 0", "id"], inplace=True, errors="ignore")
df_LR.fillna(df.median(numeric_only=True), inplace=True)


for col in df.select_dtypes(include='object').columns:
    df[col] = LabelEncoder().fit_transform(df[col].astype(str))


X = df_LR.drop(columns=["satisfaction"])
y = df["satisfaction"]


scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)




model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)


print("📊 Classification Report:")
print(classification_report(y_test, y_pred))


feature_names = df.drop(columns=["satisfaction"]).columns
importances = model.feature_importances_
sorted_idx = importances.argsort()[::-1]

plt.figure(figsize=(8, 5))
bars = plt.barh(range(len(sorted_idx[:10])), importances[sorted_idx[:10]][::-1], color='mediumseagreen')
plt.yticks(range(len(sorted_idx[:10])), feature_names[sorted_idx[:10]][::-1])
plt.xlabel("Feature Importance")
plt.title("Top 10 Feature Importances (XGBoost)")
for i, v in enumerate(importances[sorted_idx[:10]][::-1]):
    plt.text(v + 0.002, i, f"{v:.3f}", va='center')
plt.tight_layout()
plt.show()

**XG Boost insights**

Both the logistic regression and XGBoost models identify similar core drivers of airline satisfaction, but they highlight them in different ways due to their distinct model architectures.

Shared Key Predictors:
Online Boarding, Inflight Wifi Service, and Type of Travel appear among the top features in both models, confirming their strong influence across both linear and non-linear patterns.

XGBoost Insights:
XGBoost places dominant weight on Online Boarding (importance = 0.44), showing that a smooth digital check-in experience is a game-changer for customers.

It also captures complex interactions and non-linear effects, giving visibility to features like Class, Seat Comfort, and Inflight Entertainment.

Logistic Regression Insights:
Logistic regression provides clear directionality — for instance, Arrival Delay and Departure Time Convenience have negative coefficients, highlighting that delays strongly reduce satisfaction.

This model is more interpretable, helping us understand how each feature moves the probability of satisfaction up or down.

 Summary:
XGBoost excels at prediction accuracy and capturing hidden patterns.

Logistic Regression is ideal for explanation and hypothesis testing.

Using both in combination allows us to balance predictive power with interpretability, and cross-validate which features consistently matter most to customers.

### 4.3.4 Decision Tree Visualization Trial

In [ ]:
# Visualize the top 5 layers of the best model
plt.figure(figsize=(20, 10))
plot_tree(
    best_model,  # Trained decision tree model
    feature_names=X_train.columns,
    class_names=["Not Satisfied", "Satisfied"],  # You can also use ["0", "1"]
    filled=True,
    max_depth=5  # Limit the visualization to the top 5 layers
)
plt.title("Decision Tree Visualization (Top 5 Layers)")
plt.show()


# List key decision nodes and their conditions
def list_top_feature_nodes(model, feature_names, top_n_features=15, max_depth=5, min_samples=50):
    from sklearn.tree import _tree

    tree = model.tree_

    # Get the top N most important features
    importances = model.feature_importances_
    top_features = set(np.array(feature_names)[np.argsort(importances)[-top_n_features:]])

    print(f"Total nodes: {tree.node_count}")
    print(f"Filtering: Top {top_n_features} important features + samples ≥ {min_samples} + depth ≤ {max_depth}\n")

    for node in range(tree.node_count):
        if tree.feature[node] != _tree.TREE_UNDEFINED:
            feature = feature_names[tree.feature[node]]
            threshold = tree.threshold[node]
            depth = get_node_depth(tree, node)
            samples = tree.n_node_samples[node]

            if feature in top_features and depth <= max_depth and samples >= min_samples:
                left = tree.children_left[node]
                right = tree.children_right[node]
                print(f" Node {node} | Depth {depth}")
                print(f" Feature used: {feature}")
                print(f" Condition: if {feature} <= {threshold:.4f} -> go to left node {left}; else -> right node {right}")
                print(f" Sample count: {samples}")
                print("-" * 40)

# Helper function: Get the depth level of a node
def get_node_depth(tree, node_id):
    depth = 0
    current = node_id
    while current != 0:
        parent = np.where((tree.children_left == current) | (tree.children_right == current))[0]
        if len(parent) == 0:
            break
        current = parent[0]
        depth += 1
    return depth


list_top_feature_nodes(best_model, feature_names=X_train.columns.tolist())

# Plot the top 15 most important features
importances = pd


In [ ]:
def print_decision_paths(model, feature_names, class_names=None, max_depth=5, min_samples=30):
    from sklearn.tree import _tree

    tree = model.tree_
    paths = []

    def recurse(node, path, depth):
        if depth > max_depth:
            return
        if tree.feature[node] != _tree.TREE_UNDEFINED:
            name = feature_names[tree.feature[node]]
            threshold = tree.threshold[node]
            # Left child
            recurse(tree.children_left[node],
                    path + [f"{name} ≤ {threshold:.2f}"], depth + 1)
            # Right child
            recurse(tree.children_right[node],
                    path + [f"{name} > {threshold:.2f}"], depth + 1)
        else:
            value = tree.value[node].flatten()
            predicted_class = int(np.argmax(value))
            samples = tree.n_node_samples[node]
            if samples >= min_samples:
                paths.append((path, predicted_class, samples))

    recurse(0, [], 0)

    # Output Path
    for i, (rules, cls, samples) in enumerate(paths, 1):
        print(f"\n📌 Path {i}:")
        for rule in rules:
            print(f"  - {rule}")
        label = class_names[cls] if class_names else str(cls)
        print(f"➡️  Predict: {label} | Samples: {samples}")


In [ ]:
print_decision_paths(
    best_model,
    feature_names=X_train.columns.tolist(),
    class_names=["Not Satisfied", "Satisfied"],  # 可选
    max_depth=5,  # Top 5 layers
    min_samples=50  # Avoid less samples
)


Decision Path Analysis

We analyzed the internal structure of the trained decision tree and extracted rule-based paths from the root node to each leaf. These decision paths represent the model’s logic for determining whether a passenger is likely to be satisfied or dissatisfied.

Path 1: Even when online boarding and Wi-Fi scores are low, higher cleanliness leads to positive satisfaction for 1,424 passengers. Clean cabin conditions can compensate for weaker digital experience.

Path 2: Covers over 15,000 samples. When online boarding and Wi-Fi are average, combined with non-business travel and lower class, dissatisfaction dominates. These customers require basic service reinforcement and optimiazation.

Paths 3, 4, 7: Show a consistent pattern where business travelers with strong Wi-Fi and digital services tend to be satisfied. This segment expects premium consistency.

Path 6: Although online boarding is good, inconsistent Wi-Fi thresholds lead to dissatisfaction. Stability matters more than just average ratings.



Recommendations:

Prioritize improving online check-in and inflight Wi-Fi experience;

Focus on essentials for leisure/economy passengers (cleanliness, comfort);

Maintain premium Wi-Fi and inflight service for business travelers

## 4.4 Random Forest

### 4.4.1 Basic Model of Random Forest

In [ ]:
train_randomforest = df.copy()
test_randomforest = df_test.copy()

# Delete 'id' and 'Age' columns from train and test
train = train_randomforest.drop(columns=['id', 'Age'], errors='ignore')
test = test_randomforest.drop(columns=['id', 'Age'], errors='ignore')

# Encoding strings in each attribute
label_encoders = {}
for col in train.select_dtypes(include='object').columns:
    le = LabelEncoder()
    train[col] = le.fit_transform(train[col].astype(str))
    if col in test.columns:
        test[col] = le.transform(test[col].astype(str))
    label_encoders[col] = le

#assign values to the training variable
target_column = 'satisfaction'
X_train = train.drop(columns=[target_column])
y_train = train[target_column]

X_val = test.drop(columns=[target_column])
y_val = test[target_column]

In [ ]:
#Basic Random Forest model
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_val)

print("Random Forest Accuracy:", accuracy_score(y_val, y_pred_rf))
print("Classification Report (Random Forest):\n", classification_report(y_val, y_pred_rf))


### 4.4.2 Previous Colleagues' Random Forest Model Trial Running

In [ ]:
numerical_features = ['Flight Distance', 'Departure Delay in Minutes', 'Arrival Delay in Minutes']
categorical_features = ['Gender', 'Customer Type', 'Type of Travel', 'Class']

# Create a numerical transformer (Impute + Scale)
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),  # fit int/float
    ('scaler', StandardScaler())
])

# Create a categorical transformer (Impute)
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),  # ✅ fit strings
])

# Combine transformers using ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ])

In [ ]:
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])
param_dist = {
     'classifier__n_estimators': randint(50, 200),  # Random numbers fro 50-200 are chosen
     'classifier__max_depth': [None, 10, 20, 30],  # Choose from these fixed values
     'classifier__min_samples_split': randint(1, 10),  # Randomly choose between 2 and 10
     'classifier__min_samples_leaf': randint(1, 10),  # Randomly choose between 1 and 5
     'classifier__criterion': ['gini', 'entropy'],  # Criterion for splitting nodes
     'classifier__max_features': ['auto', 'sqrt', 'log2'],  # Choose one of these options
     'classifier__bootstrap': [True, False]  # Choose either True or False
}

# Instantiate RandomizedSearchCV
random_search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_dist,
    n_iter=10,  # Number of random combinations to try
    cv=5,  # Number of cross-validation folds
    scoring='accuracy',  # Scoring method
    n_jobs=-1,  # Use all available cores
    verbose=1,  # Show progress of search
    random_state=42  # Set random state for reproducibility
)

# Fit RandomizedSearchCV
random_search.fit(X_train, y_train)

# Get the best model
best_model_rf = random_search.best_estimator_

best_params_rf = random_search.best_params_

y_pred_rf = best_model_rf.predict(X_val)

In [ ]:
print(" Best Parameters Found:")
print(best_params_rf)

print("\n Accuracy on Test Set:", accuracy_score(y_val, y_pred_rf))
print("\n Confusion Matrix:")
print(confusion_matrix(y_val, y_pred_rf))

print("\n Classification Report:")
print(classification_report(y_val, y_pred_rf, target_names=["Not Satisfied", "Satisfied"]))

In [ ]:
importances = best_model_rf.named_steps['classifier'].feature_importances_
feature_names = best_model_rf.named_steps['preprocessor'].get_feature_names_out()
importances_df = pd.Series(importances, index=feature_names).sort_values(ascending=False)

importances_df.head(15).plot(kind='barh', figsize=(10,6), title="Top 15 Feature Importances")
plt.gca().invert_yaxis()
plt.show()

This random forest model is from Yifei' coding. The code from him has been tried to run and the results are shown above. However, the accuracy of his model is only 79%, which means this model didn't fit the data quite well. So we decide us general RandomForestClassifier to create the model and use GridSearchCV to optimiaze the parameter in the model.

### 4.4.3 Final Tuned Random Forest Model & Feature Insights

4.4.3.1 model evaluation and feature importance exploration

In [ ]:
# GridSearch Parameters
param_grid = {
    'n_estimators': [100],
    'max_depth': [5, 10, None],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'criterion': ['gini', 'entropy']
}

# model training
clf = RandomForestClassifier(random_state=42)
grid_search = GridSearchCV(clf, param_grid, cv=5, scoring='roc_auc', n_jobs=-1, verbose=1)
grid_search.fit(X_train, y_train)

# Optimal modelling and forecasting
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_val)
y_prob = best_model.predict_proba(X_val)[:, 1]

# model evaluation
print("Best Parameters:", grid_search.best_params_)
print("Accuracy:", accuracy_score(y_val, y_pred))
print("ROC AUC:", roc_auc_score(y_val, y_prob))
print("Confusion Matrix:\n", confusion_matrix(y_val, y_pred))
print("Classification Report:\n", classification_report(y_val, y_pred))

# Feature Importance Chart
importances = pd.Series(best_model.feature_importances_, index=X_train.columns)
top_features = importances.sort_values(ascending=False).head(15)

plt.figure(figsize=(10, 6))
top_features.plot(kind='barh')
plt.title("Top 15 Feature Importances (Random Forest)")
plt.xlabel("Importance")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

# Actual vs Probability
plt.figure(figsize=(6,6))
plt.scatter(y_val, y_prob, c=y_val, cmap='coolwarm', alpha=0.6)
plt.plot([0, 1], [0, 1], 'r--')
plt.xlabel("Actual")
plt.ylabel("Predicted Probability")
plt.title("Actual vs Predicted Probability (RF)")
plt.grid(True)
plt.tight_layout()
plt.show()

# ROC
fpr, tpr, thresholds = roc_curve(y_val, y_prob)
plt.figure(figsize=(6,6))
plt.plot(fpr, tpr, label=f'AUC = {roc_auc_score(y_val, y_prob):.2f}')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve (Random Forest)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

4.4.3.2 Confusion Matrix of Random Forest tuned model

In [ ]:
# Confusion Matrix
plt.figure(figsize=(6,4))
sns.heatmap(confusion_matrix(y_val, y_pred), annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

Model evaluation for Random Forest

The tuned Random Forest model demonstrated outstanding classification performance, achieving an overall accuracy of 96% and a macro-averaged F1 score of 0.96. Specifically, the model achieved a precision of 0.97 and recall of 0.94 for the "Satisfied" class, indicating its strong capability to both correctly identify satisfied users and avoid false positives. The ROC AUC score reached an exceptional 0.9939, confirming the model’s excellent separability between classes across all thresholds. Compared with baseline model, hyperparameter optimization makes the model superior generalization and robustness.



Model Comparison between Decision Tree and Random Forest

The tuned Random Forest has the highest accuracy (96.5%), while the base Random Forest also reach 96.1% and the tuned Decision Tree reach 95.8%.

The performance shows that the integrated model is better than the single tree model, and the tuned model has stronger application in reality.

The tuned Random Forest model outperforms all other models in nearly every metric, particularly in identifying satisfied customers with high precision (0.97) and recall (0.94), and an excellent ROC AUC of 0.9939. That indicates that Random Forest is better at avoiding wrong recognition of satisfied customers.

The tuned Decision Tree model also shows improvement over the baseline, but remains slightly behind the Random Forest model in performance. Therefore, for high-stakes classification like customer satisfaction prediction, Random Forest with parameter tuning is the recommended model due to its robustness, accuracy, and low errors.


**4.4.3.3 General Insights**

Model Insights

In general sights from Decision Tree and Random Forest, The recall rate on "satisfied customers" is significantly higher for both models (DT = 0.92, RF = 0.94) after hyperparameter tuning and training, suggesting that the predictive model is more effective in identifying people with satisfaction, which helps make out loyalty campaign and membership management. The high precision of satisfied passengers (0.97) indicates that the system is less likely to misidentify "dissatisfied" customers as "satisfied", thus avoiding wastage of resources or customer resentment due to misplacement of offers. It is critical to help marketing teams to deliver the incentive coupons and measures to the right passengers whose satisfaction needs to be improved in case of the waste of resources and money.


Business Insights from Random Forest

Online boarding (correlation ≈ ~0.5) is significantly and positively correlated with satisfaction. Although the correlation coefficient of online boarding is moderate (correlation ≈ ~0.28), it is one of the most important predictor variables in the model, suggesting a possible non-linear or interaction effect, highly captured by the random forest.

To maximize their effectiveness, airlines should prioritize the optimization of the end-to-end digital onboarding experience, from intuitive user interfaces to timely check-in notifications. In parallel, ensuring reliable and fast in-flight WIFI access can greatly enhance perceived value, particularly for business travelers and frequent flyers. Together, these digital service touchpoints offer a high-return opportunity to strengthen customer satisfaction and brand loyalty.

Business travel (correlation ≈ 0.45) and Business class (correlation ≈ 0.50) are also identified as highly influential factors in determining passenger satisfaction. Unlike service-related variables, these attributes are largely shaped by passengers’ travel intent and purchasing choices.

Given their significant correlation, these factors indicate that passengers traveling for business or flying in business class generally report higher satisfaction levels. Therefore, offering class upgrade opportunities when capacity allows, especially for loyal or early-check-in passengers, can serve as an effective strategy to boost satisfaction while also optimizing seat utilization.

Seat Comfort, Leg Room Service, and Cleanliness all exhibit moderate to strong positive correlations with satisfaction (correlation coefficients around 0.3), indicating that passengers do care about physical comfort and environmental hygiene. However, their relatively low feature importance in the Random Forest model suggests that these variables are not strong differentiators in predicting satisfaction, possibly because their effects are partially absorbed by other high-impact features such as Class or Type of Travel.

While they may not be top predictive drivers, these factors remain critical for delivering consistent service quality. As such, they should be viewed as part of a long-term operational improvement strategy, supporting brand reputation, customer retention, and post-flight reviews.




## 4.5 Artificial Neural Network (ANN)

### 4.5.1 Apply Basic ANN and Its Performance

In [ ]:
df_ANN = df.copy()
df_test_ANN = df_test.copy()

# Label encoding target variable in df_ANN train set
df_ANN['satisfaction'] = df_ANN['satisfaction'].map({'satisfied': 1, 'neutral or dissatisfied': 0})

# Feature Encoding
cat_cols = df_ANN.select_dtypes(include=['object']).columns
df_encoded = df_ANN.copy()

le = LabelEncoder()
for col in cat_cols:
    df_encoded[col] = le.fit_transform(df_ANN[col])



# Label encoding target variable in df_test_ANN test set
df_test_ANN['satisfaction'] = df_test_ANN['satisfaction'].map({'satisfied': 1, 'neutral or dissatisfied': 0})

# Feature Encoding
cat_cols = df_test_ANN.select_dtypes(include=['object']).columns
df_test_encoded = df_test_ANN.copy()

le = LabelEncoder()
for col in cat_cols:
    df_test_encoded[col] = le.fit_transform(df_test_ANN[col])


X_train = df_encoded.drop('satisfaction', axis = 1)
y_train = df_encoded['satisfaction']

X_test = df_test_encoded.drop('satisfaction', axis = 1)
y_test = df_test_encoded['satisfaction']

# Standardized numerical characteristics (optional)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
#Plot colors
target_colors=['#f44f49', '#49eef4']

In [ ]:
# Apply basic ANN model
model = Sequential()
model.add(Dense(64, input_dim=X_train_scaled.shape[1], activation='relu'))  # Input layer+hidden layer
model.add(Dropout(0.2))                         # Dropout Avoid overfitting
model.add(Dense(1, activation='sigmoid'))               # Output layer (for binary classification)

# Compile model
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# Model training
history = model.fit(X_train_scaled, y_train)

# Predict & Evaluate
y_pred = (model.predict(X_test_scaled) > 0.5).astype("int32")
print("✅ Accuracy:", accuracy_score(y_test, y_pred))
print("📋 Classification Report:\n", classification_report(y_test, y_pred))

### 4.5.2 ANN Tunnings

In [ ]:
# Grid parameter combination to find the best parameter combination
hidden_units_list = [32, 64]
dropout_list = [0.2, 0.3]
batch_sizes = [32, 64]
epochs_list = [20, 30]

best_acc = 0
best_params = {}

for hidden_units in hidden_units_list:
    for dropout in dropout_list:
        for batch_size in batch_sizes:
            for epochs in epochs_list:
                # Model building
                model = Sequential()
                model.add(Dense(hidden_units, activation='relu', input_shape=(X_train_scaled.shape[1],)))
                model.add(Dropout(dropout))
                model.add(Dense(hidden_units // 2, activation='relu'))
                model.add(Dropout(dropout))
                model.add(Dense(1, activation='sigmoid'))
                model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

                # Early stop setting
                early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

                # Train model
                model.fit(X_train_scaled, y_train,
                          epochs=epochs,
                          batch_size=batch_size,
                          verbose=0,
                          validation_data=(X_test_scaled, y_test),
                          callbacks=[early_stop])

                # Verify accuracy
                y_pred = model.predict(X_test_scaled)
                y_pred = (y_pred > 0.5).astype(int)
                acc = accuracy_score(y_test, y_pred)

                print(f"Try parameters: hidden={hidden_units}, dropout={dropout}, batch={batch_size}, epochs={epochs} → acc={acc:.4f}")

                # Update the optimal parameter combination
                if acc > best_acc:
                    best_acc = acc
                    best_params = {
                        'hidden_units': hidden_units,
                        'dropout': dropout,
                        'batch_size': batch_size,
                        'epochs': epochs
                    }

print("\n Best Accuracy Rate: {:.4f}".format(best_acc))
print("Best Parameter Combination:", best_params)


In [ ]:
# Use the best parameter combination to build model
model = Sequential()
model.add(Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)))
model.add(Dropout(0.2))
model.add(Dense(32, activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(1, activation='sigmoid'))  # Output layer (for binary classification)

# Compilation Model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Early stop setting
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# Model training
history = model.fit(
    X_train_scaled, y_train,
    validation_split=0.2,
    epochs=30,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

# Model prediction
y_pred = model.predict(X_test_scaled)
y_pred_classes = (y_pred > 0.5).astype(int)

# Model evalustion
print("✅ Accuracy:", accuracy_score(y_test, y_pred_classes))
print("\n📋 Classification Report:\n", classification_report(y_test, y_pred_classes))

# Confusion Matrix
plt.figure(figsize=(6,4))
sns.heatmap(confusion_matrix(y_test, y_pred_classes), annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix (ANN-full)')
plt.show()


### 4.5.3 Top 10 important Features Exploration

4.5.3.1 Reduced_ANN: without top 10 important features

In [ ]:
# Apply the data without the top 10 important features (from random forest) in ANN and manage the X_train and X_test
X_reduced_train = df_encoded.drop(['satisfaction', 'Inflight wifi service', 'Online boarding', 'Type of Travel', 'Class', 'Inflight entertainment', 'Ease of Online booking', 'Seat comfort', 'Customer Type', 'Leg room service', 'Flight Distance'], axis = 1)
X_reduced_test = df_test_encoded.drop(['satisfaction', 'Inflight wifi service', 'Online boarding', 'Type of Travel', 'Class', 'Inflight entertainment', 'Ease of Online booking', 'Seat comfort', 'Customer Type', 'Leg room service', 'Flight Distance'], axis = 1)

In [ ]:
# Grid parameter combination to find the best parameter combination
hidden_units_list = [32, 64]
dropout_list = [0.2, 0.3]
batch_sizes = [32, 64]
epochs_list = [20, 30]

best_acc = 0
best_params = {}

for hidden_units in hidden_units_list:
    for dropout in dropout_list:
        for batch_size in batch_sizes:
            for epochs in epochs_list:
                # Model building
                model = Sequential()
                model.add(Dense(hidden_units, activation='relu', input_shape=(X_reduced_train.shape[1],)))
                model.add(Dropout(dropout))
                model.add(Dense(hidden_units // 2, activation='relu'))
                model.add(Dropout(dropout))
                model.add(Dense(1, activation='sigmoid'))
                model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

                # Early stop setting
                early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

                # Train model
                model.fit(X_reduced_train, y_train,
                          epochs=epochs,
                          batch_size=batch_size,
                          verbose=0,
                          validation_data=(X_reduced_test, y_test),
                          callbacks=[early_stop])

                # Verify accuracy
                y_pred = model.predict(X_reduced_test)
                y_pred = (y_pred > 0.5).astype(int)
                acc = accuracy_score(y_test, y_pred)

                print(f"Try parameters: hidden={hidden_units}, dropout={dropout}, batch={batch_size}, epochs={epochs} → acc={acc:.4f}")

                # Update the optimal parameter combination
                if acc > best_acc:
                    best_acc = acc
                    best_params = {
                        'hidden_units': hidden_units,
                        'dropout': dropout,
                        'batch_size': batch_size,
                        'epochs': epochs
                    }

print("\n Best Accuracy Rate: {:.4f}".format(best_acc))
print("Best Parameter Combination:", best_params)


In [ ]:
# Use the best parameter combination to build model
model = Sequential()
model.add(Dense(64, activation='relu', input_shape=(X_reduced_train.shape[1],)))
model.add(Dropout(0.3))
model.add(Dense(32, activation='relu'))
model.add(Dropout(0.3))
model.add(Dense(1, activation='sigmoid'))  # Output layer (for binary classification)

# Compilation Model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Early stop setting
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# Model training
history = model.fit(
    X_reduced_train, y_train,
    validation_split=0.2,
    epochs=20,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

# Model prediction
y_pred = model.predict(X_reduced_test)
y_pred_classes = (y_pred > 0.5).astype(int)

# Model evalustion
print("✅ Accuracy:", accuracy_score(y_test, y_pred_classes))
print("\n📋 Classification Report:\n", classification_report(y_test, y_pred_classes))

# Confusion Matrix
plt.figure(figsize=(6,4))
sns.heatmap(confusion_matrix(y_test, y_pred_classes), annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix (ANN-Reduced)')
plt.show()


4.5.3.2 Top10_ANN: with only the top 10 important features

In [ ]:
# Apply the data only with the top 10 important features in ANN and manage the X_train and X_test
X_top10_train = df_encoded[['satisfaction', 'Inflight wifi service', 'Online boarding', 'Type of Travel', 'Class', 'Inflight entertainment', 'Ease of Online booking', 'Seat comfort', 'Customer Type', 'Leg room service', 'Flight Distance']]
X_top10_test = df_test_encoded[['satisfaction', 'Inflight wifi service', 'Online boarding', 'Type of Travel', 'Class', 'Inflight entertainment', 'Ease of Online booking', 'Seat comfort', 'Customer Type', 'Leg room service', 'Flight Distance']]

In [ ]:
# Grid parameter combination to find the best parameter combination
hidden_units_list = [32, 64]
dropout_list = [0.2, 0.3]
batch_sizes = [32, 64]
epochs_list = [20, 30]

best_acc = 0
best_params = {}

for hidden_units in hidden_units_list:
    for dropout in dropout_list:
        for batch_size in batch_sizes:
            for epochs in epochs_list:
                # Model building
                model = Sequential()
                model.add(Dense(hidden_units, activation='relu', input_shape=(X_top10_train.shape[1],)))
                model.add(Dropout(dropout))
                model.add(Dense(hidden_units // 2, activation='relu'))
                model.add(Dropout(dropout))
                model.add(Dense(1, activation='sigmoid'))
                model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

                # Early stop setting
                early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

                # Train model
                model.fit(X_top10_train, y_train,
                          epochs=epochs,
                          batch_size=batch_size,
                          verbose=0,
                          validation_data=(X_top10_test, y_test),
                          callbacks=[early_stop])

                # Verify accuracy
                y_pred = model.predict(X_top10_test)
                y_pred = (y_pred > 0.5).astype(int)
                acc = accuracy_score(y_test, y_pred)

                print(f"Try parameters: hidden={hidden_units}, dropout={dropout}, batch={batch_size}, epochs={epochs} → acc={acc:.4f}")

                # Update the optimal parameter combination
                if acc > best_acc:
                    best_acc = acc
                    best_params = {
                        'hidden_units': hidden_units,
                        'dropout': dropout,
                        'batch_size': batch_size,
                        'epochs': epochs
                    }

print("\n Best Accuracy Rate: {:.4f}".format(best_acc))
print("Best Parameter Combination:", best_params)


In [ ]:
# Use the best parameter combination to build model
model = Sequential()
model.add(Dense(32, activation='relu', input_shape=(X_top10_train.shape[1],)))
model.add(Dropout(0.2))
model.add(Dense(16, activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(1, activation='sigmoid'))  # Output layer (for binary classification)

# Compilation Model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Early stop setting
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# Model training
history = model.fit(
    X_top10_train, y_train,
    validation_split=0.2,
    epochs=20,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

# Model prediction
y_pred = model.predict(X_top10_test)
y_pred_classes = (y_pred > 0.5).astype(int)

# Model evalustion
print("✅ Accuracy:", accuracy_score(y_test, y_pred_classes))
print("\n📋 Classification Report:\n", classification_report(y_test, y_pred_classes))

# Confusion Matrix
plt.figure(figsize=(6,4))
sns.heatmap(confusion_matrix(y_test, y_pred_classes), annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix (ANN-Top10)')
plt.show()


In [ ]:
# Loss curve
plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

# Accuracy curve
plt.subplot(1,2,2)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.title('Accuracy over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.tight_layout()
plt.show()


### 4.5.4 Insights of ANN

Insights of Artificial Neural Network

We also tried Artificial Neural Network to analyze the data and manually applied grid search to find the best parameter combination to fit the model. It is an attempt to capture the extremely complex and subtle satisfaction impact patterns that may exist in order to achieve the highest theoretical prediction accuracy. It indeed has very high accuracy (0.96) and AUC score (0.99), indicating that ANN is a strong model to predict the result accurately. The precision rate of satisfied passengers (0.97) is slightly higher than the precision rate of dissatisfied passengers (0.95) while the recall rate of dissatisfied passengers (0.98) is higher than the recall rate of satisfied passengers (0.94). This may indicate that ANN model is good at predicting satisfied passengers and is less likely to miss the prediction of dissatisfied passengers.

To further explore how the top 10 important features (from Random Forest) affect the classification result, we also tried two other ANN models, one without the top 10 features (reduced_ANN) and one with only the top 10 features (top10_ANN). The bad accuracy of the reduced_ANN (), indicating that the top 10 features occupy most of the effective information and the model has lost its primary predictive basis, resulting in a decrease in performance. But there is one interesting thing that the accuracy of top10_ANN is always 100% with different parameter combinations. With the epoches increasing, the loss is decreasing to 0 and the accuracy is increasing to 100%. The possible reason could be the top 10 features are of great significance to affect the result and the model overfit the dataset with only these features. So it is necessary to have all of the features when applying ANN to avoid neither overfit nor inaccurate and get a balanced result.

Thus, the excellent performance of ANN indicates that satisfaction is determined by the interaction of complex and some nonlinear factors. From a business perspective, the linear improvement of a single service may not be sufficient, and attention needs to be paid to the overall experience and the synergistic effects between factors.

But the decision-making process of ANN is like a 'black box' which means it is extremely difficult to explain. It has high demand on data volume, computing resources, time-consuming training and tuning. In business, it is necessary to trust the predictive results of the model in some situations even if it cannot fully explain its internal logic. A trade-off needs to be made between model performance and interpretability.In conclusion, ANN is suitable for situations where prediction accuracy is required to the extreme with sufficient data, computing resources and professional knowledge to support model development and maintenance. It is suitable for automation systems that require extremely high precision, rather than decision support that requires detailed explanations of the reasons. ANN is not the best model to analyze this dataset since our purpose is to identify which factors most significantly influence whether a passenger is 'satisfied' or 'neutral or dissatisfied'.


## 4.6 Support Vector Machines(SVM)

### 4.6.1 Evaluate Model Performance

In [ ]:
df_SVM = df.copy()
df_test_SVM = df_test.copy()


#  Label encoding target variable in df_SVM train set
df_SVM['satisfaction'] = df_SVM['satisfaction'].map({'satisfied': 1, 'neutral or dissatisfied': 0})
df_test_SVM['satisfaction'] = df_test_SVM['satisfaction'].map({'satisfied': 1, 'neutral or dissatisfied': 0})


label_encoders = {}
for col in df_SVM.select_dtypes(include=['object']).columns:
    le = LabelEncoder()
    df_SVM[col] = le.fit_transform(df_SVM[col])
    df_test_SVM[col] = le.transform(df_test_SVM[col])
    label_encoders[col] = le
X_train = df_SVM.drop(columns=['satisfaction'])
y_train = df_SVM['satisfaction']
X_test = df_test_SVM.drop(columns=['satisfaction'])
y_test = df_test_SVM['satisfaction']

# Standardized features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Training SVM
svm_model = SVC(kernel="rbf", random_state=42)
svm_model.fit(X_train_scaled, y_train)


# Calculate the accuracy
accuracy = svm_model.score(X_test_scaled, y_test)
accuracy

In [ ]:
df_SVM = df.copy()
df_test_SVM = df_test.copy()


# Label encoding target variable in df_SVM train set
df_SVM['satisfaction'] = df_SVM['satisfaction'].map({'satisfied': 1, 'neutral or dissatisfied': 0})
df_test_SVM['satisfaction'] = df_test_SVM['satisfaction'].map({'satisfied': 1, 'neutral or dissatisfied': 0})


label_encoders = {}
for col in df_SVM.select_dtypes(include=['object']).columns:
    le = LabelEncoder()
    df_SVM[col] = le.fit_transform(df_SVM[col])
    df_test_SVM[col] = le.transform(df_test_SVM[col])
    label_encoders[col] = le

X_train = df_SVM.drop(columns=['satisfaction'])
y_train = df_SVM['satisfaction']
X_test = df_test_SVM.drop(columns=['satisfaction'])
y_test = df_test_SVM['satisfaction']

# Standardized features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Training SVM
svm_model = SVC(kernel="rbf", random_state=42)
svm_model.fit(X_train_scaled, y_train)
svm_rbf = SVC(kernel="rbf", probability=True)
svm_rbf.fit(X_train_scaled, y_train)

# Prediction and evaluation
y_pred = svm_rbf.predict(X_test_scaled)
y_proba = svm_rbf.predict_proba(X_test_scaled)[:, 1]

print("Classification Report:\n")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:\n")
print(confusion_matrix(y_test, y_pred))

# Visualization
plt.figure(figsize=(12, 5))

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_proba)
roc_auc = auc(fpr, tpr)
plt.subplot(1, 2, 1)
plt.plot(fpr, tpr, color="darkorange", lw=2, label=f"ROC curve (AUC = {roc_auc:.2f})")
plt.plot([0, 1], [0, 1], color="navy", lw=2, linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - RBF SVM")
plt.legend(loc="lower right")

# PR Curve
precision, recall, _ = precision_recall_curve(y_test, y_proba)
plt.subplot(1, 2, 2)
plt.plot(recall, precision, color="green", lw=2)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve - RBF SVM")

plt.tight_layout()
plt.show()


The ROC is accurate in distinguishing between satisfied and unsatisfied passengers, which could be seen from the area under the ROC curve (AUC) of 0.99. In addition, the green line of Precision-Recall Curve is mostly parallel to the level of Precision = 1. Moreover, due to the high precision, it shows a highly stationary trend, indicating that the model maintains a high  accuracy in prediction without losing the recall rate. According to both of the models, there is rarely misjudgment and it is excellent at identifying satisfied and dissatisfied passengers.

### 4.6.2 Feature Importance Exploration

In [ ]:
df_SVM = df.copy()


label_encoders = {}
for col in df_SVM.select_dtypes(include="object").columns:
    le = LabelEncoder()
    df_SVM[col] = le.fit_transform(df_SVM[col])
    label_encoders[col] = le

# Check the existence of class
if "Class" not in df_SVM.columns:
    raise KeyError("The 'Class' column is not found in the dataset.")

# Gaining business code
business_code = label_encoders["Class"].transform(["Business"])[0]
df_SVM["Class_Grouped"] = df_SVM["Class"].apply(lambda x: "Business" if x == business_code else "Non-Business")

# Start to build the model in group
for class_group in ["Business", "Non-Business"]:
    df_SVM_sub = df_SVM[df_SVM["Class_Grouped"] == class_group]

    X = df_SVM_sub.drop(columns=["satisfaction", "Class", "Class_Grouped", "id", "Gender"], errors="ignore")
    y = df_SVM_sub["satisfaction"]

    for col in X.select_dtypes(include="object").columns:
        X[col] = LabelEncoder().fit_transform(X[col])


    X = X.dropna()
    y = y.loc[X.index]

    if X.empty:
        print(f"Warning: No data in group {class_group} after dropna(). Skipping...")
        continue

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    svm = SVC(kernel="linear", random_state=42)
    svm.fit(X_scaled, y)

    coef = pd.Series(svm.coef_[0], index=X.columns).sort_values()

    # Visialization
    plt.figure(figsize=(8, 6))
    coef.plot(kind="barh")
    plt.title(f"Linear SVM Feature Importance - {class_group}")
    plt.xlabel("Coefficient")
    plt.tight_layout()
    plt.savefig(f"linear_svm_feature_importance_{class_group}.png")
    plt.show()

In the Linear SVM feature importance, it could be seen that there are obvious differences in the factors influencing satisfaction among different passenger groups. For non-business passengers, the most critical factor is Inflight wifi service, which has a much higher positive weight than other features. It indicates that this group of passengers are concerned more about digital enterntainment services during the journey. On the other hand, most of business passengers would concentrate on the experience of Online boarding、Checkin service、On-board service and Cleanliness, demonstrating that they have higher demands for boading and a clean environment. Furthermore, passengers of both business and non-business flights are easily to feel unsatisfied on Type of Travel and Customer Type because the coefficients of two services are strong negative. Moreover, Arrival Delay in Minutes and Ease of Online booking also show consistent negative impacts, which illustrates that the efficiency and streamline should also be improved.

In "Inflight wifi service", passengers in support vectors generally give the ratings between 3 and 4, non-support vectors have overall low scores, which is between 2 and 3. For Inflight wifi service, the “support vector” passengers, which the model relies on heavily, scored higher than the average passenger. Therefore, better WiFi ratings do not automatically indicate higher overall passenger satisfaction. According to the "Online boarding", it could be seen that the medium numbers of both box plots are 4.0. The compactness of the green box of "online boarding" is more centralized than the green box of "Inflight wifi", indicating that their ratings are more consistent. However, non-support vectors are discrete, which include 1 to 5. The support vectors of both checkin service and seat comfort are discrete, which are in the range of 0 to 5,but the area of the box for non-supper vectors of checkin service is much smaller than seat comfort.

Overall, after comparing the box plots for four services ratings, it could be summarised that fluctuations in the ratings of support vector passengers are more concentrated in the ambiguous rating area, while non-support vector passengers tend to be at the extremes of the model. Passengers who give the rates between 2 to 4 and would not express their satisfaction or dissatisfaction explicitly are easily seen as support vectors. This group of passengers belongs to the intermediate group of users that the model is hard to define. The satisfaction of the support vector could be enhanced with a small push, it is possible that a small improvement in one of the crucial points of contact could transform them into the satisfied group. For example, sending surveys to support vector groups, and establish a point system to earn rewards for a certain amount of points. It would increase the engagement of passengers and give passengers the perception that their feedback is valued and acted upon.


### 4.6.3 SVM Confusion Matrix


In [ ]:
df_SVM = df.copy()
df_test_SVM = df_test.copy()


# Label encoding target variable in df_SVM train set
df_SVM['satisfaction'] = df_SVM['satisfaction'].map({'satisfied': 1, 'neutral or dissatisfied': 0})
df_test_SVM['satisfaction'] = df_test_SVM['satisfaction'].map({'satisfied': 1, 'neutral or dissatisfied': 0})

label_encoders = {}
for col in df_SVM.select_dtypes(include=['object']).columns:
    le = LabelEncoder()
    df_SVM[col] = le.fit_transform(df_SVM[col])
    df_test_SVM[col] = le.transform(df_test_SVM[col])
    label_encoders[col] = le

X_train = df_SVM.drop(columns=['satisfaction'])
y_train = df_SVM['satisfaction']
X_test = df_test_SVM.drop(columns=['satisfaction'])
y_test = df_test_SVM['satisfaction']

# Standardized features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# training
svm_model = SVC(kernel="rbf", random_state=42)
svm_model.fit(X_train_scaled, y_train)

y_pred = svm_model.predict(X_test_scaled)



#DataFrame
result_df = pd.DataFrame({
    "True Label": y_test,
    "Predicted Label": y_pred
})
result_df["Correct"] = result_df["True Label"] == result_df["Predicted Label"]
result_df["Correct Label"] = result_df["Correct"].map({True: "Correct", False: "Wrong"})


plot_data = result_df.groupby(["True Label", "Correct Label"]).size().unstack(fill_value=0)

#Stacked Bar
plot_data.plot(kind="bar", stacked=True, figsize=(8, 6), color=["red", "lightblue"])
plt.title("Prediction Accuracy per True Label (Stacked Bar)")
plt.xlabel("True Satisfaction Label (0 = Not Satisfied, 1 = Satisfied)")
plt.ylabel("Number of Passengers")
plt.legend(title="Prediction Result")
plt.tight_layout()
plt.show()

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=["Dissatisfied", "Satisfied"],
            yticklabels=["Dissatisfied", "Satisfied"])
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("SVM Confusion Matrix")
plt.tight_layout()

plt.show()



The prediction accuracy stacked bar chart and SVM confusion Matrix are used to compare the predictive satisfaction and actual satisfaction of passengers in order to demonstrate the accuracy. The Stacked Bar shows that the model predicted both satisfied passengers (1) and dissatisfied passengers (0) accurately because the red portion is dominant. Based on the confusion matrix, the model has successfully recognized 11314 dissatisfied passengers and 8474 satisfied passengers. Nevertheless, it could be seen that 590 satisfied passengers were incorrectly classified as unsatisfied and 341 unsatisfied passengers were predicted to be satisfied. The model probably makes judgement relying on a part of rates of passengers, which should be support vectors. Thus, the model is not applicable in forecasting the satisfaction of passengers whose ratings are close to the midpoint.


### 4.6.4 SVM Tunnings

In [ ]:
# Best params of C and Gamma
params = {
    "C": [0.1, 1, 10],
    "gamma": [0.001, 0.01, 0.1]
}


svm = SVC(kernel="rbf")
grid = GridSearchCV(svm, params, scoring="f1", cv=5)
grid.fit(X_train_scaled, y_train)


print("Best params:", grid.best_params_)


results = grid.cv_results_
scores = results["mean_test_score"]
param_C = [d["C"] for d in results["params"]]
param_gamma = [d["gamma"] for d in results["params"]]

heatmap_data = pd.DataFrame({
    "C": param_C,
    "gamma": param_gamma,
    "F1": scores
})
heatmap_pivot = heatmap_data.pivot(index="gamma", columns="C", values="F1")

# Visualizing heat map
plt.figure(figsize=(8, 6))
sns.heatmap(heatmap_pivot, annot=True, fmt=".3f", cmap="YlGnBu")
plt.title("GridSearchCV F1 Score - SVM (RBF Kernel)")
plt.ylabel("Gamma")
plt.xlabel("C")
plt.tight_layout()
plt.show()

The horizontal axis (c) indicates the strength of the penalty for misclassification. The higher the C value represents the more perfect the classification, but it would be overfitting. The vertical axis (Gamma)  determines the influence range of the single sample, since the gamma is larger, the model would be more sensitive to the change of data. As a result, the model recognizes that the optimal values for C and gamma are 1 and 0.1 respectively with the highest standardised score of 0.948 in the heatmap, which would not be overfitting or underfitting.
From the opinion of business, the optimal combination (1, 0.1) represents that the model could accurately recognize the support vectors and tolerate the misclassification moderately. In contrast, excessively large C-values, low C-values, and low Gamma-values might make the inaccuracy of the model increase. If C is 10.0, the model would recognize the support vectors as "satisfied" or "unsatisfied" based on the rate of a single service, despite the passenger has positively rated another service. A simultaneously small C and gamma may lead to ineffective model judgment,which would result in a reduction in the predictive power of customer satisfaction, meanwhile losing some of the support vectors.


### 4.6.5 Optimal Features Exploration and Insights

In [ ]:
df_SVM = df.copy()
df_test_SVM = df_test.copy()


# Label encoding target variable in df_SVM train set
df_SVM['satisfaction'] = df_SVM['satisfaction'].map({'satisfied': 1, 'neutral or dissatisfied': 0})
df_test_SVM['satisfaction'] = df_test_SVM['satisfaction'].map({'satisfied': 1, 'neutral or dissatisfied': 0})

label_encoders = {}
for col in df_SVM.select_dtypes(include=['object']).columns:
    le = LabelEncoder()
    df_SVM[col] = le.fit_transform(df_SVM[col])
    df_test_SVM[col] = le.transform(df_test_SVM[col])
    label_encoders[col] = le

X_train = df_SVM.drop(columns=['satisfaction'])
y_train = df_SVM['satisfaction']
X_test = df_test_SVM.drop(columns=['satisfaction'])
y_test = df_test_SVM['satisfaction']

# Standardized features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# SVM training
svm = SVC(kernel="rbf")
svm.fit(X_train_scaled, y_train)

# Extract support vectors
support_indices = svm.support_
df_SVM_support = df_SVM.iloc[support_indices].copy()
df_SVM_support["Support Vector"] = "Yes"
df_SVM_non_support = df_SVM.drop(index=df_SVM_support.index).copy()
df_SVM_non_support["Support Vector"] = "No"

# Combination
df_SVM_compare = pd.concat([df_SVM_support, df_SVM_non_support])

# Visualization
key_scores = ["Inflight wifi service", "Online boarding", "Checkin service", "Seat comfort"]

plt.figure(figsize=(14, 10))
for i, col in enumerate(key_scores):
    plt.subplot(2, 2, i+1)
    sns.boxplot(data=df_SVM_compare, x="Support Vector", y=col, palette="Set2")
    plt.title(f"{col} (Support Vector vs Others)")
plt.tight_layout()
plt.show()

In "Inflight wifi service", passengers in support vectors generally give the ratings between 3 and 4, non-support vectors have overall low scores, which is between 2 and 3. For Inflight wifi service, the “support vector” passengers, which the model relies on heavily, scored higher than the average passenger. Therefore, better WiFi ratings do not automatically indicate higher overall passenger satisfaction. According to the "Online boarding", it could be seen that the medium numbers of both box plots are 4.0. The compactness of the green box of "online boarding" is more centralized than the green box of "Inflight wifi", indicating that their ratings are more consistent. However, non-support vectors are discrete, which include 1 to 5. The support vectors of both checkin service and seat comfort are discrete, which are in the range of 0 to 5,but the area of the box for non-supper vectors of checkin service is much smaller than seat comfort.

Overall, after comparing the box plots for four services ratings, it could be summarised that fluctuations in the ratings of support vector passengers are more concentrated in the ambiguous rating area, while non-support vector passengers tend to be at the extremes of the model. Passengers who give the rates between 2 to 4 and would not express their satisfaction or dissatisfaction explicitly are easily seen as support vectors. This group of passengers belongs to the intermediate group of users that the model is hard to define. The satisfaction of the support vector could be enhanced with a small push, it is possible that a small improvement in one of the crucial points of contact could transform them into the satisfied group. For example, sending surveys to support vector groups, and establish a point system to earn rewards for a certain amount of points. It would increase the engagement of passengers and give passengers the perception that their feedback is valued and acted upon.

# 5. **Conclusion**

## 5.1 Model Insights

In summary, the performance and applicability of multiple classification models are evaluated. Random Forest has the best performance with the highest accuracy of 0.9650 and remarkable generalization ability. It is suitable in business deployment. Despite the complexity of the model structure, the stability and business suitability lead it to be one of the preferred models.

XGBoost has an extremely strong prediction ability with an accuracy of 0.9627, which could automatically accomplish the feature selections, especially in processing the large scale data set. Moreover, it is adaptable in both linear and non-linear relationships. The disadvantage of XGBoost is the long training time and weak interpretability.

Similarity,  Artificial Neutral Network(ANN) and Support Vector Machine(SVM) also have an high accuracy . ANN has an excellent performance of handle complex data. SVM could classify the groups in the medium. However, both of them suffer from difficulty in tuning parameters and excessive computational resource requirements. Besides, the models have the inferiority in interpretability. Despite the Naive Bayes and Logistic Regression models are efficient in calculation and simple to implement, they have relatively low accuracy and are difficult to meet the high requirements for customer satisfaction classification accuracy.


##5.2 Business Insights

In the first place,  it is recommended that the airline should optimize online check-in process, inflight wi-fi seivice, seat comfort and cabin cleanliness, especially in the middle and latter part of the journey (aligning with Peak End Rule). Furtherly it could enhance the perception of passengers about the service experience.
Secondly, it is recommended that marginal customers with ratings between 2 and 4 be targeted for conversion. This type of passenger is located between satisfied and dissatisfied. Moreover, it is the target group with the greatest potential for improvement. Light incentives such as targeted questionnaires, points or small gifts could be used, along with fine-tuning the service process, to achieve effective satisfaction conversion.

Third, stratified service improvement could be implemented for the phenomenon of low satisfaction of economy class and private class. By analyzing the demand structure in depth, differentiated services would satisfy the demands and preferences of diverse passengers group.

Fourth, from an overall  perspective, airline should promote service synergy is essencial. Optimizing customer journeys in a systematic way single-point optimization, enhancing coordination across all segments, and incorporating structural factors such as seating and cleaning into a chronic plan.

In addition, intelligent customer retention strategies could be deployed by combining multiple predictive models such as logistic regression and XGBoost. For example, using LR models to flag potentially dissatisfied customers in advance and proactively send apology emails or surveys. Moreover, airline could  distributing Wi-Fi or snack coupons to dissatisfied customers. Training models could be continuously updated to dynamically respond to changes in customer behavior by integrating new data generated by the feedback system on a monthly basis.


# Reference

Sutton, J. (2019). *What Is the Peak End Rule and How to Use It Smartly*. [online] PositivePsychology.com, accessed 20 April 2025, <https://positivepsychology.com/what-is-peak-end-theory/?utm_source=chatgpt.com>.